In [ ]:
configuration = "A"
scenario = "1_1"

In [ ]:
from collections import defaultdict

import h2o
from h2o.automl import H2OAutoML

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, bernoulli, uniform, expon

from scenarios import *
from utils import *

In [ ]:
configurations=dict(
    A=dict(max_runtime_secs=360,  max_models=20),
    B=dict(max_runtime_secs=720,  max_models=20),
    C=dict(max_runtime_secs=1440, max_models=20),
    D=dict(max_runtime_secs=2880, max_models=20),
    E=dict(max_runtime_secs=360,  max_models=40),
    F=dict(max_runtime_secs=720,  max_models=40),
    G=dict(max_runtime_secs=1440, max_models=40),
    H=dict(max_runtime_secs=2880, max_models=40),
    I=dict(max_runtime_secs=360,  max_models=80),
    J=dict(max_runtime_secs=720,  max_models=80),
    K=dict(max_runtime_secs=1440, max_models=80),
    L=dict(max_runtime_secs=2880, max_models=80),
    M=dict(max_runtime_secs=360,  max_models=160),
    N=dict(max_runtime_secs=720,  max_models=160),
    O=dict(max_runtime_secs=1440, max_models=160),
    P=dict(max_runtime_secs=2880, max_models=160))

aml_config = configurations[configuration]

n = 10_000

rng = np.random.default_rng(42)
_gt_data =  dict(
        X1 = norm.rvs(size=n, loc=0, scale=1, random_state=rng),
        X2 = bernoulli.rvs(size=n, p=0.5, random_state=rng),
        X3 = norm.rvs(size=n, loc=0, scale=1, random_state=rng),
        X4 = uniform.rvs(size=n, loc=-2, scale=4, random_state=rng),
        X5 = expon.rvs(size=n, scale=1, random_state=rng),
        X6 = rng.choice(['A','B','C'], size=n, p=[1/3]*3),
        X7 = norm.rvs(size=n, loc=0, scale=1, random_state=rng),
        X8 = rng.choice(['0','1'], size=n, p=[1/2]*2),
        X9 = norm.rvs(size=n, loc=0, scale=1, random_state=rng))


def new_data(k, v):
    res = dict(**_gt_data)
    res[k] = np.full(n, v)
    return res

def scenario_1_pdp(**kwargs):
    def f(X1,X2,X3,X4,X5,X6,X7,X8,X9):
        return 2.9*X1 - 3.7*X2+1.2*X3+0.8*X1*X1-1.5*X2*X3+1.1*X4-0.5*np.log(X5+1) + \
                np.where(X6 == "B", 2, np.where(X6=="C", -1, 0)) + 0.3*X9
        
    var = list(kwargs.keys())[0]
    return[np.mean(f(**new_data(var, val))) for val in kwargs[var]]

def scenario_2_pdp(**kwargs):
    def f(X1,X2,X3,X4,X5,X6,X7,X8,X9):
        return np.sin(X1) + np.log(np.abs(X3)+1)+0.8*X4*X5 - 1.2*X1*X3 + np.where(X6 == "A", -2, np.where(X6=="C", 1, 0)) + \
             1.5*X7+np.where(X8 == "1", 2, 0) + 0.4*X9*X9
        
    var = list(kwargs.keys())[0]
    return[np.mean(f(**new_data(var, val))) for val in kwargs[var]]

def scenario_3_pdp(**kwargs):
    def f(X1,X2,X3,X4,X5,X6,X7,X8,X9):
        return X1 + 2*X2+0.5*np.exp(-(X3*X3))+0.7*np.sin(np.pi*X4)+0.6*X5**0.3+np.where(X6=='C',3,0)+\
             0.9*X7+1.2*X9
        
    var = list(kwargs.keys())[0]
    return[np.mean(f(**new_data(var, val))) for val in kwargs[var]]

def gt_pdp(**kwargs):
    if scenario[0] == "1":
        f = gt_scenario_1
    elif scenario[0] == "2":
        f = gt_scenario_2
    elif scenario[0] == "3":
        f = gt_scenario_3
    else:
        raise RuntimeError(f"Ooops. Scenario {scenario} has no gt_pdp defined.")
    var = list(kwargs.keys())[0]
    return[np.mean(f(**new_data(var, val))) for val in kwargs[var]]


In [ ]:
h2o.init()

In [ ]:
frame =  h2o.import_file(f"data/synth_{scenario}.csv")
frame["X8"] = frame["X8"].asfactor()
frame.types

In [ ]:
test_frame =  h2o.import_file(f"data/test_synth_{scenario}.csv")
test_frame["X8"] = test_frame["X8"].asfactor()
test_frame.types

In [ ]:
aml = H2OAutoML(seed=0xC0FFEE, **aml_config)
aml.train(y="Y", training_frame=frame)
aml.leaderboard

In [ ]:
h2o.make_leaderboard(aml, test_frame)

In [ ]:
for col in [f"X{i}" for i in range(1, 10)]:
    if test_frame.types[col] == "enum":
        continue
    aml.pd_multi_plot(test_frame, col)
    x = np.linspace(*plt.xlim(), 100)
    y = gt_pdp(**{col: x})
    suffix = f" (constant = {np.mean(y):.4f})" if np.abs(np.std(y)) < 1e-9 else ""
    plt.plot(x, y, c="k", label="Ground Truth"+suffix, ls="--")
    plt.legend()
    plt.show()
    

In [ ]:
for col in [f"X{i}" for i in range(1, 10)]:
    if test_frame.types[col] == "enum":
        continue
    aml.pd_multi_plot(test_frame, col, best_of_family=False)
    x = np.linspace(*plt.xlim(), 100)
    y = gt_pdp(**{col: x})
    suffix = f" (constant = {np.mean(y):.4f})" if np.abs(np.std(y)) < 1e-9 else ""
    plt.plot(x, y, c="k", label="Ground Truth"+suffix, ls="--")
    plt.legend()
    plt.show()

In [ ]:
leaderboard = aml.leaderboard.as_data_frame()
leaderboard

In [ ]:
header("Rashomon models")
rashomon_models = get_rashomon_set(leaderboard, "rmse", False, eps=0.05)

header("Best of Family models")
best_of_family_models = [m.model_id for m in [aml.get_best_model(m) for m in ["deeplearning", "drf", "gbm", "glm", "stackedensemble", "xgboost"]] if m]
display(leaderboard[leaderboard.model_id.isin(best_of_family_models)])

header("Best of Family models ∩ Rashomon models")
bof_rashomon = list(set(rashomon_models).intersection(set(best_of_family_models)))
display(leaderboard[leaderboard.model_id.isin(bof_rashomon)])

all_models = leaderboard["model_id"].values
h2o.no_progress()
results = defaultdict(list)
for col in  [f"X{i}" for i in range(1, 10)]:
    header(col, "=")
    for desc, models in [
        ("Best model", all_models[:1]),
        ("All models", all_models),
        ("Rashomon", rashomon_models), 
        ("Best of Family", best_of_family_models),
        ("Best of Family models ∩ Rashomon models", bof_rashomon)
                        ]:
        res = [h2o.get_model(model).partial_plot(test_frame, cols=[col], nbins=100, plot=False)[0].as_data_frame() for model in models]
        mean_pdp = np.mean([r["mean_response"] for r in res ], axis=0)
        x = res[0][col.lower()]
        gt = gt_pdp(**{col:x})
        md = mean_distance_to_gt_pdp(mean_pdp, gt)
        results[desc].append(md)
        
        plt.figure(figsize=(16,9))
        plt.plot(x, mean_pdp, c="blue", label="Avg. PDP")
        plt.plot(x, gt, c="green", label="Ground Truth")
        plt.grid()
        plt.legend()
        plt.title(f"{desc}; {col}, Mean Distance to GT PDP = {md}")
        plt.show()


header("Mean Distance to the GT PDP")
for desc, vals in results.items():
    print(f"{desc}: {np.mean(vals)} ± {np.std(vals)}")

In [ ]:
pd.DataFrame(results).to_csv(f"results/mean_distance_to_gt_pdp_{configuration}_{scenario}.csv", index=False)

In [ ]:
h2o.cluster().shutdown()